In [1]:
import warnings
#warnings.simplefilter(action='ignore')
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from pycaret.utils import version
version()
import scipy

# Compatibilità temporanea con librerie che importano scipy.interp
if not hasattr(scipy, "interp"):
    scipy.interp = np.interp

## Best models

In [2]:
import pandas as pd
fold_col = "fold"           # Nome della colonna con fold_1, ..., fold_5
id_col = "patient_id"

df_train = pd.read_csv("folds_100/train_STATIC+BiPadLSTM_0.csv")
df_test = pd.read_csv("folds_100/test_STATIC+BiPadLSTM_0.csv")
assert set(df_train[fold_col]) == set(df_test[fold_col])

In [3]:
import pandas as pd
from pycaret.classification import ClassificationExperiment
from sklearn.metrics import brier_score_loss

fold_col = "fold"
id_col = "patient_id"
target = "target"

if df_train[fold_col].isna().any() or df_test[fold_col].isna().any():
    raise ValueError("Sono presenti fold mancanti.")

if set(df_train[fold_col]) != set(df_test[fold_col]):
    raise ValueError("I fold del train e del test non coincidono.")

from sklearn.metrics import confusion_matrix, precision_recall_curve, auc
from IPython.display import display

# Classificazione binaria: 0 = negativo, 1 = positivo.
def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    # NPV non definito se non ci sono predizioni negative.
    return tn / (tn + fn) if (tn + fn) > 0 else float("nan")


def auprc_score(y_true, y_score):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    return auc(recall, precision)

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return tn / (tn + fp) if (tn + fp) > 0 else float("nan")

risultati = []

for fold_id in df_train[fold_col].unique():
    print(f"Elaborazione fold: {fold_id}", flush=True)

    train_fold = df_train.loc[
        df_train[fold_col] == fold_id
    ].copy()

    test_fold = df_test.loc[
        df_test[fold_col] == fold_id
    ].copy()

    # Controlla che i pazienti del test non siano nel train.
    if train_fold[id_col].isna().any() or test_fold[id_col].isna().any():
        raise ValueError(f"Patient ID mancanti nel fold {fold_id}.")

    overlap = set(train_fold[id_col]) & set(test_fold[id_col])
    if overlap:
        raise ValueError(
            f"Fold {fold_id}: {len(overlap)} pazienti "
            "presenti sia nel train sia nel test."
        )

    # Fold e identificativo non devono essere feature.
    train_fold = train_fold.drop(
        columns=[fold_col, id_col]
    ).reset_index(drop=True)

    test_fold = test_fold.drop(
        columns=[fold_col, id_col]
    ).reset_index(drop=True)

    exp = ClassificationExperiment()

    exp.setup(
        data=train_fold,
        target=target,
        test_data=test_fold,
        index=False,
        session_id=123,
        use_gpu=False,
        fix_imbalance=True,
        verbose=False,
        html=False,
    )

    exp.add_metric(
        id="brier",
        name="Brier Score",
        score_func=brier_score_loss,
        target="pred_proba",
        greater_is_better=False,
    )

    exp.add_metric(
        id="npv",
        name="NPV",
        score_func=npv_score,
        target="pred",
        greater_is_better=True,
    )
    exp.add_metric(
        id="auprc",
        name="AUPRC",
        score_func=auprc_score,
        target="pred_proba",
        greater_is_better=True,
    )
    exp.compare_models(
        cross_validation=False,  # Valuta sul test_fold fornito
        sort="MCC",
        round=3,
        errors="raise",
        verbose=False,
    )
    exp.add_metric(
        id="specificity",
        name="specificity",
        score_func=specificity_score,
        target="pred",
        greater_is_better=True,
    )    
    tabella = exp.pull().copy()
    tabella["fold"] = fold_id
    risultati.append(tabella)

    print(f"Risultati fold {fold_id}")
    with pd.option_context("display.max_columns", None):
        display(tabella)
    
    tabella = exp.pull().copy()
    tabella["fold"] = fold_id
    risultati.append(tabella)

    print(f"Fold {fold_id} completato.", flush=True)

# Risultati di tutti i modelli in tutti i fold.
risultati_fold = pd.concat(risultati, ignore_index=True)
risultati_fold

Elaborazione fold: 1
Risultati fold 1


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Brier Score,NPV,AUPRC,TT (Sec),fold
gbc,Gradient Boosting Classifier,0.914,0.846,0.60,0.536,0.566,0.519,0.520,0.070,0.958,0.587,35.39,1
svm,SVM - Linear Kernel,0.907,0.787,0.64,0.500,0.561,0.510,0.515,0.093,0.962,0.587,0.11,1
lr,Logistic Regression,0.877,0.866,0.76,0.413,0.535,0.471,0.500,0.082,0.973,0.599,0.29,1
ada,Ada Boost Classifier,0.907,0.827,0.60,0.500,0.546,0.494,0.496,0.178,0.958,0.472,6.59,1
lightgbm,Light Gradient Boosting Machine,0.910,0.870,0.56,0.518,0.538,0.489,0.489,0.080,0.954,0.613,4.67,1
xgboost,Extreme Gradient Boosting,0.907,0.853,0.56,0.500,0.528,0.477,0.478,0.078,0.954,0.630,13.52,1
et,Extra Trees Classifier,0.892,0.837,0.64,0.444,0.525,0.466,0.476,0.088,0.961,0.451,0.17,1
knn,K Neighbors Classifier,0.892,0.839,0.60,0.441,0.508,0.449,0.456,0.080,0.957,0.581,0.08,1
ridge,Ridge Classifier,0.840,0.822,0.80,0.345,0.482,0.404,0.454,0.160,0.976,0.582,0.10,1
dt,Decision Tree Classifier,0.903,0.750,0.52,0.482,0.500,0.446,0.447,0.094,0.950,0.535,1.02,1


Fold 1 completato.
Elaborazione fold: 2
Risultati fold 2


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Brier Score,NPV,AUPRC,TT (Sec),fold
xgboost,Extreme Gradient Boosting,0.925,0.921,0.731,0.594,0.655,0.614,0.618,0.069,0.970,0.692,2.86,2
gbc,Gradient Boosting Classifier,0.922,0.909,0.731,0.576,0.644,0.601,0.606,0.068,0.970,0.758,35.49,2
lightgbm,Light Gradient Boosting Machine,0.918,0.895,0.692,0.562,0.621,0.575,0.579,0.072,0.966,0.660,3.46,2
et,Extra Trees Classifier,0.899,0.853,0.769,0.488,0.597,0.543,0.561,0.083,0.974,0.516,0.17,2
knn,K Neighbors Classifier,0.881,0.882,0.808,0.438,0.568,0.505,0.537,0.091,0.977,0.685,0.07,2
dt,Decision Tree Classifier,0.903,0.809,0.692,0.500,0.581,0.527,0.536,0.097,0.966,0.611,0.91,2
rf,Random Forest Classifier,0.892,0.885,0.731,0.463,0.567,0.509,0.526,0.082,0.969,0.527,0.24,2
svm,SVM - Linear Kernel,0.888,0.818,0.731,0.452,0.559,0.499,0.518,0.112,0.969,0.605,0.12,2
ada,Ada Boost Classifier,0.896,0.905,0.692,0.474,0.562,0.505,0.517,0.196,0.965,0.596,6.77,2
ridge,Ridge Classifier,0.828,0.854,0.885,0.348,0.500,0.419,0.486,0.172,0.985,0.622,0.09,2


Fold 2 completato.
Elaborazione fold: 3
Risultati fold 3


...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Brier Score,NPV,AUPRC,TT (Sec),fold
knn,K Neighbors Classifier,0.899,0.819,0.654,0.486,0.557,0.502,0.509,0.087,0.961,0.553,0.07,4
dt,Decision Tree Classifier,0.903,0.723,0.500,0.500,0.500,0.446,0.446,0.097,0.946,0.524,1.18,4
lightgbm,Light Gradient Boosting Machine,0.907,0.837,0.462,0.522,0.490,0.439,0.440,0.091,0.943,0.392,54.81,4
nb,Naive Bayes,0.813,0.838,0.808,0.318,0.456,0.369,0.427,0.187,0.975,0.577,0.08,4
lr,Logistic Regression,0.881,0.862,0.577,0.417,0.484,0.418,0.425,0.100,0.953,0.411,0.31,4
xgboost,Extreme Gradient Boosting,0.907,0.833,0.423,0.524,0.468,0.418,0.420,0.090,0.939,0.399,16.14,4
ada,Ada Boost Classifier,0.892,0.837,0.500,0.448,0.473,0.413,0.413,0.167,0.946,0.447,6.79,4
gbc,Gradient Boosting Classifier,0.896,0.864,0.423,0.458,0.440,0.382,0.383,0.085,0.938,0.474,34.98,4
rf,Random Forest Classifier,0.903,0.829,0.346,0.500,0.409,0.358,0.365,0.080,0.932,0.354,0.28,4
svm,SVM - Linear Kernel,0.896,0.668,0.385,0.454,0.417,0.360,0.361,0.104,0.935,0.449,0.13,4


Fold 4 completato.
Elaborazione fold: 5
Risultati fold 5


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Brier Score,NPV,AUPRC,TT (Sec),fold
ada,Ada Boost Classifier,0.903,0.898,0.846,0.500,0.629,0.577,0.603,0.173,0.982,0.576,6.66,5
knn,K Neighbors Classifier,0.899,0.875,0.846,0.489,0.620,0.566,0.595,0.094,0.982,0.638,0.08,5
gbc,Gradient Boosting Classifier,0.910,0.945,0.769,0.526,0.625,0.576,0.590,0.062,0.974,0.698,34.45,5
lightgbm,Light Gradient Boosting Machine,0.914,0.938,0.731,0.543,0.623,0.576,0.584,0.075,0.970,0.609,37.95,5
xgboost,Extreme Gradient Boosting,0.907,0.954,0.731,0.514,0.603,0.552,0.563,0.070,0.970,0.755,21.29,5
dt,Decision Tree Classifier,0.903,0.809,0.692,0.500,0.581,0.527,0.536,0.097,0.966,0.611,1.14,5
lr,Logistic Regression,0.851,0.894,0.885,0.383,0.535,0.462,0.520,0.110,0.986,0.529,0.48,5
svm,SVM - Linear Kernel,0.858,0.836,0.808,0.389,0.525,0.453,0.495,0.142,0.977,0.608,0.16,5
ridge,Ridge Classifier,0.847,0.847,0.846,0.373,0.518,0.443,0.495,0.153,0.981,0.617,0.08,5
et,Extra Trees Classifier,0.888,0.905,0.654,0.447,0.531,0.470,0.481,0.080,0.961,0.579,0.17,5


Fold 5 completato.


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Brier Score,NPV,AUPRC,TT (Sec),fold
0,Gradient Boosting Classifier,0.914,0.846,0.600,0.536,0.566,0.519,0.520,0.070,0.958,0.587,35.39,1
1,SVM - Linear Kernel,0.907,0.787,0.640,0.500,0.561,0.510,0.515,0.093,0.962,0.587,0.11,1
2,Logistic Regression,0.877,0.866,0.760,0.413,0.535,0.471,0.500,0.082,0.973,0.599,0.29,1
3,Ada Boost Classifier,0.907,0.827,0.600,0.500,0.546,0.494,0.496,0.178,0.958,0.472,6.59,1
4,Light Gradient Boosting Machine,0.910,0.870,0.560,0.518,0.538,0.489,0.489,0.080,0.954,0.613,4.67,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,Random Forest Classifier,0.884,0.904,0.654,0.436,0.523,0.460,0.472,0.076,0.961,0.614,0.23,5
146,Naive Bayes,0.758,0.794,0.808,0.259,0.392,0.288,0.361,0.242,0.973,0.543,0.08,5
147,Linear Discriminant Analysis,0.772,0.782,0.731,0.260,0.384,0.281,0.338,0.207,0.964,0.367,1.53,5
148,Quadratic Discriminant Analysis,0.903,0.500,0.000,0.000,0.000,0.000,0.000,0.097,0.903,0.548,3.38,5


In [5]:
risultati_fold = pd.concat(risultati, ignore_index=True)

metriche = [
    col
    for col in risultati_fold.select_dtypes(include="number").columns
    if col not in ["fold", "TT (Sec)"]
]

statistiche = (
    risultati_fold
    .groupby("Model")[metriche]
    .agg(["mean", "std"])
    .sort_values(("MCC", "mean"), ascending=False)
)

riepilogo = pd.DataFrame(index=statistiche.index)

for metrica in metriche:
    riepilogo[metrica] = [
        f"{media:.3f}±{std:.3f}" if pd.notna(std)
        else f"{media:.3f}±NA"
        for media, std in zip(
            statistiche[(metrica, "mean")],
            statistiche[(metrica, "std")],
        )
    ]

riepilogo = riepilogo.rename(columns={
    "Prec.": "PPV",
    "Recall": "Sensitivity",
    "Brier Score": "Brier"
}).drop(columns=["Kappa"])
ncols = ["AUC", "AUPRC", "F1", "PPV", "Sensitivity", "NPV", "MCC", "Accuracy", "Brier"]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(riepilogo[ncols])

,AUC,AUPRC,F1,PPV,Sensitivity,NPV,MCC,Accuracy,Brier
Model,,,,,,,,,
Light Gradient Boosting Machine,0.878±0.038,0.562±0.099,0.563±0.054,0.526±0.027,0.612±0.101,0.958±0.010,0.517±0.059,0.910±0.007,0.082±0.009
K Neighbors Classifier,0.851±0.025,0.598±0.060,0.555±0.041,0.455±0.029,0.720±0.098,0.968±0.010,0.514±0.052,0.890±0.010,0.090±0.006
Extreme Gradient Boosting,0.888±0.047,0.595±0.136,0.559±0.068,0.523±0.040,0.612±0.122,0.958±0.012,0.514±0.073,0.909±0.009,0.079±0.009
Gradient Boosting Classifier,0.888±0.037,0.615±0.107,0.557±0.079,0.506±0.056,0.628±0.128,0.959±0.013,0.511±0.088,0.905±0.014,0.074±0.010
Ada Boost Classifier,0.863±0.034,0.514±0.063,0.543±0.056,0.467±0.034,0.658±0.120,0.962±0.012,0.497±0.067,0.895±0.011,0.176±0.011
SVM - Linear Kernel,0.771±0.063,0.551±0.067,0.503±0.062,0.430±0.054,0.636±0.151,0.960±0.015,0.456±0.070,0.881±0.022,0.119±0.022
Decision Tree Classifier,0.754±0.054,0.539±0.076,0.507±0.080,0.460±0.076,0.573±0.104,0.953±0.011,0.453±0.091,0.893±0.022,0.107±0.022
Logistic Regression,0.862±0.027,0.504±0.080,0.488±0.045,0.369±0.049,0.744±0.104,0.970±0.011,0.450±0.054,0.849±0.029,0.108±0.017
Random Forest Classifier,0.852±0.038,0.469±0.096,0.492±0.056,0.442±0.039,0.582±0.136,0.954±0.013,0.442±0.058,0.887±0.010,0.085±0.008


In [8]:
def create_table_with_splits(formatted_results, nsplit=4):
    """Split columns into balanced blocks and stack them vertically."""
    if not isinstance(nsplit, int) or nsplit < 1:
        raise ValueError("nsplit must be a positive integer.")
    if formatted_results.shape[1] == 0:
        raise ValueError("The DataFrame must have at least one column.")

    # Avoid empty blocks
    n = len(formatted_results.columns)
    nsplit = min(nsplit, n)

    base, remainder = divmod(n, nsplit)
    sizes = [base + (i < remainder) for i in range(nsplit)]

    blocks = []
    start = 0
    for size in sizes:
        blocks.append(
            formatted_results.iloc[:, start:start + size].copy()
        )
        start += size

    width = max(sizes)
    final_columns = ["Model"] + blocks[0].columns.tolist()

    parts = []
    for i, block in enumerate(blocks):
        if i > 0:
            header = (
                ["Model"]
                + block.columns.tolist()
                + [""] * (width - block.shape[1])
            )
            parts.append(
                pd.DataFrame([header], columns=final_columns)
            )

        rows = block.rename_axis("Model").reset_index()
        rows.columns = final_columns[:rows.shape[1]]
        rows = rows.reindex(columns=final_columns, fill_value="")
        parts.append(rows)

    return pd.concat(parts, ignore_index=True)

In [15]:
from scripts.utils import highlight_best_values
table = create_table_with_splits(riepilogo, nsplit=2)
table.set_index("Model", inplace=True)
print(highlight_best_values(table.to_latex()))

\begin{tabular}{llllll}
\toprule
 & Accuracy & AUC & Sensitivity & PPV & F1 \\
Model &  &  &  &  &  \\
\midrule
Light Gradient Boosting Machine & \textbf{0.910}±0.007 & 0.878±0.038 & 0.612±0.101 & 0.526±0.027 & \textbf{0.563}±0.054 \\
K Neighbors Classifier & 0.890±0.010 & 0.851±0.025 & 0.720±0.098 & 0.455±0.029 & 0.555±0.041 \\
Extreme Gradient Boosting & 0.909±0.009 & \textbf{0.888}±0.047 & 0.612±0.122 & 0.523±0.040 & 0.559±0.068 \\
Gradient Boosting Classifier & 0.905±0.014 & \textbf{0.888}±0.037 & 0.628±0.128 & 0.506±0.056 & 0.557±0.079 \\
Ada Boost Classifier & 0.895±0.011 & 0.863±0.034 & 0.658±0.120 & 0.467±0.034 & 0.543±0.056 \\
SVM - Linear Kernel & 0.881±0.022 & 0.771±0.063 & 0.636±0.151 & 0.430±0.054 & 0.503±0.062 \\
Decision Tree Classifier & 0.893±0.022 & 0.754±0.054 & 0.573±0.104 & 0.460±0.076 & 0.507±0.080 \\
Logistic Regression & 0.849±0.029 & 0.862±0.027 & 0.744±0.104 & 0.369±0.049 & 0.488±0.045 \\
Random Forest Classifier & 0.887±0.010 & 0.852±0.038 & 0.582±0.136 & 0.4